In [1]:
# %%
import wandb
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import os

C:\Users\Bunny和Evan\AppData\Roaming\Python\Python39\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\Bunny和Evan\AppData\Roaming\Python\Python39\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


ModuleNotFoundError: No module named 'datasets'

In [ ]:
# 1. 初始化 wandb
# -------------------------------
# 请将 entity 替换为你的 wandb 用户名或团队名称
# 直接传入 API key 登录
wandb.login(key="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")
#wandb.init(project="bert-base-chinese-training", entity="19802966223-peking-university")
wandb.init(project="bert-base-chinese-training")

In [ ]:
# 2. 加载 CSV 数据，并划分训练集和验证集
# -------------------------------
# 假设 CSV 文件名称为 clean_data.csv，且文件中有两列：Review（评论文本）和 Rating（评分：1、2、3、4）
# 注意：如果 CSV 中的标签为 1,2,3,4，建议将其转为 0-indexed（例如：0,1,2,3）
data_files = {"data": "clean_data.csv"}
raw_dataset = load_dataset("csv", data_files=data_files)["data"]

In [ ]:
# %%
def process_example(example):
    # 将评分转换为 0-index（即 1->0, 2->1, 3->2, 4->3）
    example["label"] = int(example["Rating"]) - 1
    # 将评论文本存入 text 字段（方便后续 tokenize），假设评论列名为 "Review"
    example["text"] = example["Review"]
    return example

In [ ]:
# %%
# 对整个数据集进行预处理
processed_dataset = raw_dataset.map(process_example)

In [ ]:
# %%
# 划分训练集和验证集（例如 80% 训练，20% 验证）
split_dataset = processed_dataset.train_test_split(test_size=0.2, seed=42)

In [ ]:
# 3. Tokenization
# -------------------------------
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

In [ ]:
# 设置数据格式，确保返回的字段为模型输入（input_ids, attention_mask）和标签 label
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# %%

In [ ]:
# 4. 加载预训练模型
# -------------------------------
# 因为新标签数为 4（0,1,2,3），故设置 num_labels=4
model = BertForSequenceClassification.from_pretrained("bert-base-chinese", num_labels=4)

# %%

In [ ]:
print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print("Total number of parameters:", total_params)

In [ ]:
# 5. 定义 TrainingArguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./results",              # 模型和检查点保存目录
    num_train_epochs=8,                  # 总共训练 4 个 epoch
    per_device_train_batch_size=64,      # 每个设备上的训练 batch size
    per_device_eval_batch_size=64,       # 每个设备上的验证 batch size
    logging_steps=10,       
    evaluation_strategy="epoch",         # 每个 epoch 进行一次评估
    save_strategy="epoch",               # 每个 epoch 保存一次模型检查点
    logging_dir="./logs",                # 日志保存目录
    report_to="wandb",                   # 使用 wandb 记录日志
    load_best_model_at_end=False,        # 如果需要根据评估指标加载最佳模型可以设置为 True
)


In [ ]:
# 使用 DataCollatorWithPadding 自动进行 batch 内 padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    # eval_pred 包含 (logits, labels)
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
# 6. 构建 Trainer 并训练
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# 开始训练，训练过程中每个 epoch 会保存检查点并通过 wandb 记录日志
trainer.train()

# %%
# 训练结束后，可以保存最终模型
trainer.save_model(os.path.join(training_args.output_dir, "final_model"))
